# Bullseye — Phase 2 reusable YOLO26 export (₹0 workflow)

Runs **YOLO26 Nano** on **one original basketball video** using a free Google Colab session (GPU if offered; CPU otherwise), exports **every frame's candidate boxes**, then downloads a ZIP that the local Bullseye importer validates.

**No always-on server.** Colab's free compute is not guaranteed; don't buy compute units or enable paid resources. **No local YOLO26 installation.** Keep your existing local ball detector and shot tracker.

**How to use:** Runtime → Change runtime type → T4 GPU *if free* → Run all → upload any original `.mp4` file → download the generated `Bullseye_YOLO26_<video>_results.zip`. Then run `python tools/import_yolo26.py data/videos/<video>.mp4 "<zip path>" --run` inside `backend`.

In [ ]:
%pip -q install 'ultralytics>=8.4.160'
import cv2, torch, ultralytics
print("Ultralytics:", ultralytics.__version__)
print("OpenCV:", cv2.__version__)
print("Accelerator:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slower)")

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one original basketball .mp4 video.")
video_path = Path(next(iter(uploaded))).resolve()
if video_path.suffix.lower() != ".mp4":
    raise ValueError("Please upload an .mp4 file.")
print("Selected:", video_path.name)

In [ ]:
import hashlib
import json
import time
import zipfile
from ultralytics import YOLO

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

model = YOLO("yolo26n.pt")
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {video_path.name}")

fps = float(cap.get(cv2.CAP_PROP_FPS))
estimated_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if estimated_count <= 0 or width <= 0 or height <= 0:
    cap.release()
    raise RuntimeError("Video metadata unavailable.")

print(f"Video: {video_path.name}, {width}x{height}, {estimated_count} frames")
print("Computing original video fingerprint...")
video_hash = sha256_file(video_path)
print("SHA-256:", video_hash)

out_dir = Path("bullseye_yolo26_export")
out_dir.mkdir(exist_ok=True)
# Six evenly spaced diagnostic frames; no video-specific hardcoded indices.
sample_frames = {
    max(1, min(estimated_count, round(fraction * estimated_count)))
    for fraction in (0.05, 0.20, 0.40, 0.60, 0.80, 0.95)
}
records = []
person_frames = ball_frames = frame_no = 0
start = time.perf_counter()

try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_no += 1
        prediction = model.predict(
            frame,
            imgsz=640,
            conf=0.25,
            classes=[0, 32],  # COCO classes: person, sports ball
            verbose=False
        )[0]
        detections = []
        for box in prediction.boxes:
            cid = int(box.cls[0].item())
            detections.append({
                "class_id": cid,
                "class_name": "person" if cid == 0 else "sports ball",
                "confidence": float(box.conf[0].item()),
                "bbox_xyxy": [float(x) for x in box.xyxy[0].tolist()],
            })
        person_frames += any(d["class_id"] == 0 for d in detections)
        ball_frames += any(d["class_id"] == 32 for d in detections)
        records.append({"frame": frame_no, "detections": detections})

        if frame_no in sample_frames:
            cv2.imwrite(str(out_dir / f"frame_{frame_no}.jpg"), prediction.plot())
        if frame_no % 100 == 0:
            print(f"Processed {frame_no}/{estimated_count} frames")
finally:
    cap.release()

if frame_no != estimated_count:
    print(f"NOTE: video reports {estimated_count} frames but {frame_no} were readable.")
elapsed = round(time.perf_counter() - start, 2)
report = {
    "schema_version": 2,
    "model": "yolo26n.pt",
    "video": {
        "filename": video_path.name,
        "frame_count": frame_no,
        "fps": fps,
        "width": width,
        "height": height,
        "sha256": video_hash,
    },
    "frames": records,
}
summary = {
    "processed_frames": frame_no,
    "frames_with_person_candidate": person_frames,
    "frames_with_sports_ball_candidate": ball_frames,
    "elapsed_seconds": elapsed,
    "note": "Candidates can contain false detections. Detection counts are not accuracy.",
}
(out_dir / "detections.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
(out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

safe_stem = "".join(c if c.isalnum() or c in "-_" else "_" for c in video_path.stem)
zip_path = Path(f"Bullseye_YOLO26_{safe_stem}_results.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for candidate in out_dir.iterdir():
        if candidate.is_file():
            archive.write(candidate, arcname=candidate.name)
print("Summary:", json.dumps(summary, indent=2))
print("Export ready:", zip_path)

In [ ]:
from google.colab import files
files.download(str(zip_path))